# UI-Mate-9B Q6_K development checks
Select **GPU T4 x2** and **Internet on**; attach `vembody-kaggle.zip` as a private input dataset, then run these cells in order.
Optional: attach one previously exported `vembody-runtime-t4.zip` as a private input to skip compatible builds.
Weights/build files stay in `/kaggle/temp`; results and a weights-free runtime cache go into `/kaggle/working`. No public endpoint or paid service is used.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile

archives = list(Path("/kaggle/input").rglob("vembody-kaggle.zip"))
manifests = list(Path("/kaggle/input").rglob("bundle-manifest.json"))
if len(archives) == 1 and not manifests:
    archive = archives[0]
    snapshot = hashlib.sha256(archive.read_bytes()).hexdigest()
    destination = Path("/kaggle/temp") / ("vembody-bundle-" + snapshot[:12])
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as bundle:
        if any(
            not (destination / item.filename).resolve().is_relative_to(destination)
            for item in bundle.infolist()
        ):
            raise RuntimeError("Unsafe archive member")
        bundle.extractall(destination)
    REPO = destination / "vembody"
elif len(manifests) == 1 and not archives:
    snapshot = hashlib.sha256(manifests[0].read_bytes()).hexdigest()
    REPO = Path("/kaggle/temp") / ("vembody-bundle-" + snapshot[:12]) / "vembody"
    if not REPO.exists():
        shutil.copytree(manifests[0].parent, REPO)
else:
    raise RuntimeError(
        "Attach exactly one Vembody bundle; both ZIP and Kaggle-unpacked inputs are supported"
    )
manifest = json.loads((REPO / "bundle-manifest.json").read_text())
for name, expected in manifest["files"].items():
    if not (REPO / name).resolve().is_relative_to(REPO):
        raise RuntimeError("Unsafe manifest path")
    if hashlib.sha256((REPO / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Bundle checksum mismatch: " + name)
OUTPUT = Path("/kaggle/working") / ("vembody-q6-" + uuid4().hex[:12])
print("Source snapshot:", snapshot)

In [ ]:
sys.path.insert(0, str(REPO / "scripts"))
sys.modules.pop("kaggle_environment", None)
from kaggle_environment import create_environment  # noqa: E402 - load this verified bundle first.

print("Loaded setup:", create_environment.__code__.co_filename)
ENV = Path("/kaggle/temp") / ("vembody-env-" + snapshot[:12])
PYTHON = create_environment(REPO, ENV)
WORK = Path("/kaggle/temp/ui-mate-q6-runtime")
RUNNER = REPO / "scripts/kaggle_run.py"
MODE = "autonomous"  # Alternatives: 'grounding' for four T1 tasks, 'pilot' for C5 only.
ADAPTER = "ui-mate"  # v2 contract alignment; 'ui-mate-legacy' repeats the v1 adapter.
# Controls: 'budget-only' (v2/4096 tokens), 'original' (v2/2048).
AUTONOMOUS_PROFILE = "brief"  # v3 + 4096 tokens.
TRIALS = 3  # Fixed openings and greedy decoding: repeatability, not independent samples.
PLAY_TRIALS = 1  # Each autonomous game has a 40-action/15-minute maximum.
subprocess.run(
    [
        str(PYTHON),
        "-I",
        str(REPO / "scripts/check_ui_mate_compatibility.py"),
        "--output",
        "/kaggle/working/ui-mate-compatibility.json",
    ],
    check=True,
)
print("Isolated Python:", PYTHON)

Without a compatible runtime cache, CUDA compilation may take an hour or more; weights plus FP16 projector total about 8.6 GB. Grounding runs four T1 tasks three times plus one smoke call; autonomous mode plays one existing 5x5 development game from a high-level goal. No training occurs.

In [ ]:
subprocess.run(
    [
        str(PYTHON),
        "-I",
        str(RUNNER),
        "prepare",
        "--repo",
        str(REPO),
        "--work",
        str(WORK),
        "--cache-output",
        "/kaggle/working/vembody-runtime-t4.zip",
    ],
    check=True,
)

In [ ]:
OUTPUT = Path("/kaggle/working") / (
    "vembody-q6-" + uuid4().hex[:12]
)  # Fresh on every evaluation retry.
count = PLAY_TRIALS if MODE == "autonomous" else TRIALS
subprocess.run(
    [
        str(PYTHON),
        "-I",
        str(RUNNER),
        MODE,
        "--repo",
        str(REPO),
        "--work",
        str(WORK),
        "--output",
        str(OUTPUT),
        "--trials",
        str(count),
        "--adapter",
        ADAPTER,
        "--autonomous-profile",
        AUTONOMOUS_PROFILE,
    ],
    check=True,
)

In [ ]:
import shutil

result_zip = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
print("Results ZIP:", result_zip)
print("Download it from the right sidebar: Output > /kaggle/working > file menu > Download.")
print("Also download vembody-runtime-t4.zip; attach it as a PRIVATE input in future sessions.")
print("Only after downloading results/cache, STOP the Kaggle session to stop spending GPU quota.")